# 1. IT Helpdesk: Vector RAG vs. Graph RAG — and *when* the graph actually helps

**Presentation map:** Slide 32 (vector RAG is the baseline — and where it breaks), Slide 36 (graphs help on reasoning,
not on fact lookup), Slide 38 (a decision rule for graph retrieval), Slide 43 (exposing a graph to an agent as tools)

**Dataset:** `data/aura_helpdesk.json` — a synthetic IT-helpdesk knowledge graph for the 3DS "Aura" support chatbot
(source: `AlexisBalayre/AuraHelpdeskGraph`): **23 tickets, 57 comments, 10 KB articles, 7 statuses, 10 users** and
**246 relationships** reconstructed from the original Neo4j load script.

The notebook has two parts:

1. **Walkthrough** — one incoming login ticket. A *strong* vector baseline (every ticket, comment and KB article indexed
   with a metadata header) versus a graph pipeline that uses the vector index only as an *entry point*, then walks
   `HAS_STATUS` / `CONTAINS` / `POSTS` to the comment that is marked as the verified fix.
2. **Benchmark by question type** — 22 questions with gold answers computed from the graph, in three families that
   mirror the decision rule on Slide 38: **single-hop lookup**, **verified-fix retrieval**, and **relational /
   aggregate** questions. The point is not "graph always wins" — it is *where* each method wins.

In [1]:
import json
from collections import Counter
import networkx as nx

with open("data/aura_helpdesk.json") as f:
    data = json.load(f)
print("Counts:", data["_metadata"]["counts"])

G = nx.MultiDiGraph()
for cat in data["categories"]:     G.add_node(cat["categoryId"], label="Category", **cat)
for pri in data["priorities"]:     G.add_node(pri["priorityId"], label="Priority", **pri)
for sts in data["statuses"]:       G.add_node(sts["statusId"], label="Status", **sts)
for usr in data["users"]:          G.add_node(usr["userId"], label="User", **usr)
for tkt in data["tickets"]:        G.add_node(tkt["ticketId"], label="Ticket", **tkt)
for cmt in data["comments"]:       G.add_node(cmt["commentId"], label="Comment", **cmt)
for doc in data["documentation"]:  G.add_node(doc["articleId"], label="Documentation", **doc)

# All 246 Cypher-derived relationships become typed edges
for rel in data["relationships"]:
    G.add_edge(rel["from"], rel["to"], key=rel["type"], type=rel["type"])
# tkt_011 points at cat_010, which the source data never defines -- keep it, but label it honestly
for n, d in G.nodes(data=True):
    if "label" not in d:
        d.update(label="Category", name="(undefined category)")
        print(f"Note: {n} is referenced by an edge but not defined in the source data")

print(f"\nGraph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
print("Nodes by label:", dict(Counter(nx.get_node_attributes(G, "label").values())))
print("Edges by type: ", dict(Counter(d["type"] for _, _, d in G.edges(data=True))))

# --- Small traversal helpers used throughout (each is one hop in the graph) ---
def out_nbrs(n, rtype): return [v for _, v, d in G.out_edges(n, data=True) if d["type"] == rtype]
def in_nbrs(n, rtype):  return [u for u, _, d in G.in_edges(n, data=True) if d["type"] == rtype]
def user_name(u):       return f"{G.nodes[u]['firstName']} {G.nodes[u]['lastName']}" if u in G else None
def ticket_status(t):   return G.nodes[out_nbrs(t, "HAS_STATUS")[0]]
def ticket_closed(t):   return ticket_status(t)["isClosed"] == "true"

Counts: {'categories': 9, 'priorities': 5, 'statuses': 7, 'users': 10, 'tickets': 23, 'comments': 57, 'documentation': 10, 'relationships': 246}
Note: cat_010 is referenced by an edge but not defined in the source data

Graph: 122 nodes, 246 edges
Nodes by label: {'Category': 10, 'Priority': 5, 'Status': 7, 'User': 10, 'Ticket': 23, 'Comment': 57, 'Documentation': 10}
Edges by type:  {'HAS_CATEGORY': 33, 'SUBMITS': 23, 'POSTS': 57, 'ASSIGNED_TO': 23, 'HAS_STATUS': 23, 'HAS_PRIORITY': 23, 'CONTAINS': 57, 'REFERENCED_BY': 7}


## A strong vector baseline (Slide 32: "use it first, and measure against it")

A naive baseline that embeds bare comment text is easy to beat and proves nothing. Instead we index **all 90 text
items** — 23 tickets (title, description, resolution summary), 57 comments, 10 KB articles — and prepend a
**metadata header** to each chunk (ids, status, author and role, *accepted-solution* flag). This is "contextual
chunking", a standard production trick: the vector index now *sees* the structure as text. Top-k = 8.

In [2]:
import os
import numpy as np
from openai import OpenAI

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
EMBED_MODEL, CHAT_MODEL, SEED, K = "text-embedding-3-small", "gpt-4o-mini", 7, 8

def embed(texts):
    resp = client.embeddings.create(model=EMBED_MODEL, input=texts)
    v = np.array([e.embedding for e in resp.data])
    return v / np.linalg.norm(v, axis=1, keepdims=True)

def chat(messages, **kw):
    return client.chat.completions.create(model=CHAT_MODEL, messages=messages, temperature=0, seed=SEED, **kw)

# --- 90 chunks, each with a metadata header that exposes the structure as text ---
chunks = []
for t in data["tickets"]:
    tid, st = t["ticketId"], ticket_status(t["ticketId"])
    cat = G.nodes[t["categoryId"]]["name"] if t["categoryId"] in G else "unknown"
    chunks.append({"id": tid, "label": "Ticket", "text":
        f"[Ticket {tid} | category: {cat} | status: {st['name']} | submitted by {user_name(t['createdByUserId'])} | "
        f"assigned to {user_name(t['assignedToUserId'])}]\n{t['title']}. {t['description']}\n"
        f"Resolution summary: {t['resolutionSummary'] or 'none'}"})
for c in data["comments"]:
    a = G.nodes[c["authorUserId"]]
    chunks.append({"id": c["commentId"], "label": "Comment", "text":
        f"[Comment {c['commentId']} on ticket {c['ticketId']} | by {user_name(c['authorUserId'])} ({a['role']}) | "
        f"accepted solution: {'yes' if c['isSolution'] == 'True' else 'no'}]\n{c['content']}"})
for d in data["documentation"]:
    chunks.append({"id": d["articleId"], "label": "Documentation", "text":
        f"[KB article {d['articleId']}]\n{d['title']}: {d['summary']}"})
chunk_vecs = embed([c["text"] for c in chunks])
print(f"Vector index: {len(chunks)} chunks ({Counter(c['label'] for c in chunks)})")

ANSWER_RULES = ("Answer using only the provided context. Cite the ids (tkt_..., cmt_..., doc_...) that support "
                "your answer. If the context is insufficient, say so.")

def vector_search(query, k=K, label=None):
    sims = chunk_vecs @ embed([query])[0]
    order = [i for i in np.argsort(-sims) if label is None or chunks[i]["label"] == label]
    return [(chunks[i], float(sims[i])) for i in order[:k]]

def vector_rag(question, k=K):
    hits = vector_search(question, k)
    context = "\n\n".join(c["text"] for c, _ in hits)
    answer = chat([{"role": "system", "content": "You are the Aura IT helpdesk assistant. " + ANSWER_RULES},
                   {"role": "user", "content": f"Question: {question}\n\nContext:\n{context}"}]).choices[0].message.content
    return {"answer": answer, "context_ids": [c["id"] for c, _ in hits]}

incoming_ticket = ("I cannot log into Aura with my 3DS credentials — clicking Sign In just refreshes "
                   "the login page without logging me in.")
for rank, (c, s) in enumerate(vector_search(incoming_ticket, 5), 1):
    print(f"  #{rank} [{s:.3f}] {c['id']:<8} {c['text'].splitlines()[0][:100]}")

vec_walk = vector_rag(incoming_ticket)
print("\n--- Vector RAG answer ---\n" + vec_walk["answer"])

Vector index: 90 chunks (Counter({'Comment': 57, 'Ticket': 23, 'Documentation': 10}))
  #1 [0.762] tkt_001  [Ticket tkt_001 | category: Aura Access | status: Resolved | submitted by Marie Dubois | assigned to
  #2 [0.653] doc_002  [KB article doc_002]
  #3 [0.522] tkt_008  [Ticket tkt_008 | category: Aura Access | status: In Progress | submitted by Sarah Chen | assigned t
  #4 [0.501] doc_001  [KB article doc_001]
  #5 [0.475] tkt_004  [Ticket tkt_004 | category: Aura Performance | status: Resolved | submitted by Lisa Thompson | assig



--- Vector RAG answer ---
To resolve your login issue with Aura using your 3DS credentials, I recommend clearing your browser cache and cookies. This solution was effective for another user who experienced a similar problem where the login page just refreshed without logging in (Ticket tkt_001). The issue was related to stored authentication tokens. 

If you continue to have trouble after clearing your cache and cookies, please let me know.


## Graph RAG — vector entry point + structural traversal (Slides 38, 43)

We embed only the 23 **Ticket** chunks to find the most similar *historical* ticket, then leave vector space and walk
real edges: `HAS_STATUS` (was it actually closed?) → `CONTAINS` (its comments) → keep `isSolution == "True"` →
`POSTS` (who wrote the fix, with what role). The graph answers "which of these plausible texts is the *verified* fix",
which no similarity score can.

In [3]:
def graph_verified_fix(ticket_id):
    """Template traversal: (Ticket)-[:HAS_STATUS]->(Status{isClosed}) and (Ticket)-[:CONTAINS]->(Comment{isSolution})<-[:POSTS]-(User)."""
    if not ticket_closed(ticket_id):
        return None
    for cid in out_nbrs(ticket_id, "CONTAINS"):
        if G.nodes[cid]["isSolution"] == "True":
            author = in_nbrs(cid, "POSTS")[0]
            return {"ticket": ticket_id, "comment": cid, "text": G.nodes[cid]["content"],
                    "author": user_name(author), "role": G.nodes[author]["role"]}
    return None

def graph_rag_fix(question):
    best_ticket = vector_search(question, k=1, label="Ticket")[0][0]["id"]   # vector = entry point only
    fix = graph_verified_fix(best_ticket)
    ctx = (f"Historical ticket {best_ticket} ({G.nodes[best_ticket]['title']}) was closed. Verified solution "
           f"{fix['comment']} by {fix['author']} ({fix['role']}): {fix['text']}") if fix else \
          f"Most similar ticket {best_ticket} has no verified solution yet."
    answer = chat([{"role": "system", "content": "You are the Aura IT helpdesk assistant. " + ANSWER_RULES},
                   {"role": "user", "content": f"Question: {question}\n\nVerified graph context:\n{ctx}"}]).choices[0].message.content
    return {"answer": answer, "context_ids": [best_ticket] + ([fix["comment"]] if fix else []), "fix": fix}

graph_walk = graph_rag_fix(incoming_ticket)
f_ = graph_walk["fix"]
print(f"Entry ticket: {f_['ticket']} -> status {ticket_status(f_['ticket'])['name']!r} -> verified comment {f_['comment']} "
      f"by {f_['author']} ({f_['role']})")
print("\n--- Graph RAG answer ---\n" + graph_walk["answer"])

Entry ticket: tkt_001 -> status 'Resolved' -> verified comment cmt_005 by Jean Martin (agent)

--- Graph RAG answer ---
It seems that the issue you are experiencing with logging into Aura using your 3DS credentials may be related to a browser extension interfering with the authentication process. A similar case was addressed in historical ticket tkt_001, where the solution suggested was to disable extensions one by one to identify the culprit. Additionally, using incognito mode was recommended as a temporary workaround (cmt_005).


### Side-by-side — computed, not asserted

Note the data quirk the graph exposes: `tkt_001`'s free-text `resolutionSummary` records the *first thing tried*
("clear cache"), while the comment flagged `isSolution=True` records the fix that actually worked (a browser extension).
Whether vector RAG repeats the stale advice is checked below from the actual answers.

In [4]:
gold_fix = graph_walk["fix"]["comment"]
def mentions(ans, *words): return any(w in ans.lower() for w in words)
rows = [
    ("Context items", ", ".join(vec_walk["context_ids"][:5]) + " ...", ", ".join(graph_walk["context_ids"])),
    ("Verified fix in context", gold_fix in vec_walk["context_ids"], gold_fix in graph_walk["context_ids"]),
    ("Answer mentions extensions", mentions(vec_walk["answer"], "extension"), mentions(graph_walk["answer"], "extension")),
    ("Answer suggests clearing cache", mentions(vec_walk["answer"], "cache"), mentions(graph_walk["answer"], "cache")),
]
print(f"{'Aspect':<32} | {'Vector RAG (k=8, metadata headers)':<48} | Graph RAG")
print("-" * 110)
for a, v, g in rows:
    print(f"{a:<32} | {str(v):<48} | {g}")

Aspect                           | Vector RAG (k=8, metadata headers)               | Graph RAG
--------------------------------------------------------------------------------------------------------------
Context items                    | tkt_001, doc_002, tkt_008, doc_001, tkt_004 ...  | tkt_001, cmt_005
Verified fix in context          | False                                            | True
Answer mentions extensions       | False                                            | True
Answer suggests clearing cache   | True                                             | False


## Benchmark by question type (Slides 36 and 38)

Gold answers are computed **from the graph**, never typed by hand:

| Family | n | Example | Expected winner (Slide 38) |
| :-- | :-: | :-- | :-- |
| **A. Single-hop lookup** | 5 | "Which browsers does Aura support?" → the one KB article | vector RAG (graph adds nothing) |
| **B. Verified fix** | 11 | each *closed* ticket that has an `isSolution` comment, rephrased by an LLM as a new user's report | graph — needs the status + solution flag |
| **C. Relational / aggregate** | 6 | "Who authored the most accepted solutions?" | graph — the answer is spread over many items |

The graph system here is a **tool-calling agent** (Slide 43's `ToolsRetriever` pattern): `semantic_search` over the
same 90 chunks, `find_nodes` (a filtered node scan whose rows are already joined across edges — status, assignee,
author role), `count_nodes` (aggregation done by the store, as `count(*)` in Cypher — LLMs miscount long lists),
`get_node` (one-hop neighbourhood) and the `get_verified_solution` traversal template from above.

In [5]:
from pydantic import BaseModel

# ---------- A. single-hop lookup (gold = the KB article id) ----------
A = [("Where can I check Aura's technical requirements and browser compatibility?", "doc_005"),
     ("Is there a developer guide for calling Aura programmatically?", "doc_007"),
     ("Where can I find video tutorials on using Aura?", "doc_008"),
     ("How do I learn to phrase questions so Aura gives more accurate answers?", "doc_003"),
     ("Which guide explains how to set up a new user account in 3DS systems?", "doc_010")]
questions = [{"family": "A. lookup", "q": q, "gold": {"all": [[g, G.nodes[g]["title"].lower()]]}} for q, g in A]

# ---------- B. verified fix: every closed ticket with an isSolution comment, paraphrased ----------
fix_tickets = [t for t in (n for n, d in G.nodes(data=True) if d["label"] == "Ticket") if graph_verified_fix(t)]
class Paraphrases(BaseModel):
    items: list[str]
src = "\n".join(f"{i+1}. {G.nodes[t]['title']}. {G.nodes[t]['description']}" for i, t in enumerate(fix_tickets))
para = client.chat.completions.parse(model=CHAT_MODEL, temperature=0, seed=SEED, response_format=Paraphrases, messages=[
    {"role": "user", "content": "Rewrite each support ticket below as a NEW report from a different employee: 1-2 "
     "sentences, same problem, different wording, no ids, keep the order. Return exactly one item per ticket.\n\n" + src}]
).choices[0].message.parsed.items
assert len(para) == len(fix_tickets)
for t, p in zip(fix_tickets, para):
    fx = graph_verified_fix(t)
    questions.append({"family": "B. verified fix", "q": f"New ticket: {p} What should the user do?",
                      "gold": {"comment": fx["comment"], "fix_text": fx["text"]}, "source_ticket": t})

# ---------- C. relational / aggregate (gold computed by traversal) ----------
tickets_ = [n for n, d in G.nodes(data=True) if d["label"] == "Ticket"]
users_ = [n for n, d in G.nodes(data=True) if d["label"] == "User"]
alias = lambda u: [u, user_name(u).lower()]
open_t = [t for t in tickets_ if not ticket_closed(t)]
sol_by = Counter(in_nbrs(c, "POSTS")[0] for c, d in G.nodes(data=True) if d["label"] == "Comment" and d["isSolution"] == "True")
top_solver, top_n = sol_by.most_common(1)[0]
docs_open = sorted({u for t in open_t for c in out_nbrs(t, "CONTAINS") for u in in_nbrs(c, "REFERENCED_BY")})
all_docs = [n for n, d in G.nodes(data=True) if d["label"] == "Documentation"]
access_open = [t for t in open_t if "cat_001" in out_nbrs(t, "HAS_CATEGORY")]
resp_by_sophie = [t for t in tickets_ if "cat_002" in out_nbrs(t, "HAS_CATEGORY") and in_nbrs(t, "ASSIGNED_TO") == ["usr_009"]]
jean_fixed_submitters = sorted({in_nbrs(t, "SUBMITS")[0] for t in tickets_
                                if (fx := graph_verified_fix(t)) and in_nbrs(fx["comment"], "POSTS") == ["usr_002"]})
open_by_assignee = Counter(in_nbrs(t, "ASSIGNED_TO")[0] for t in open_t)
top_assignee, top_assignee_n = open_by_assignee.most_common(1)[0]

C_ = [
    ("Which user has authored the most comments that are marked as the accepted solution, and how many?",
     {"all": [alias(top_solver), [str(top_n)]]}),
    ("Which KB articles are referenced in the comment threads of tickets that are not yet closed?",
     {"all": [[d, G.nodes[d]["title"].lower()] for d in docs_open],
      "none": [[d, G.nodes[d]["title"].lower()] for d in all_docs if d not in docs_open]}),
    ("Which 'Aura Access' tickets are not yet closed, and who is each one assigned to?",
     {"all": [[t] for t in access_open] + [alias(in_nbrs(t, "ASSIGNED_TO")[0]) for t in access_open],
      "none": [["tkt_001"]]}),
    ("How many 'Aura Responses' tickets are assigned to Sophie Laurent, and how many of those are still not closed?",
     {"all": [[str(len(resp_by_sophie))], [str(sum(not ticket_closed(t) for t in resp_by_sophie))]]}),
    ("Which users submitted tickets whose accepted solution was written by Jean Martin?",
     {"all": [alias(u) for u in jean_fixed_submitters],
      "none": [alias(u) for u in users_ if u not in jean_fixed_submitters and u != "usr_002"]}),
    ("Which person is assigned the most tickets that are still not closed, and how many?",
     {"all": [alias(top_assignee), [str(top_assignee_n)]]}),
]
questions += [{"family": "C. relational", "q": q, "gold": g} for q, g in C_]

print(Counter(q["family"] for q in questions))
for q in questions[:1] + questions[5:6] + questions[-6:]:
    print(f"- [{q['family']}] {q['q'][:110]}\n    gold: {q['gold'] if 'fix_text' not in q['gold'] else q['gold']['comment']}")

Counter({'B. verified fix': 11, 'C. relational': 6, 'A. lookup': 5})
- [A. lookup] Where can I check Aura's technical requirements and browser compatibility?
    gold: {'all': [['doc_005', 'aura system requirements']]}
- [B. verified fix] New ticket: I'm unable to access Aura as I'm encountering an error when entering my 3DS credentials. The login
    gold: cmt_005
- [C. relational] Which user has authored the most comments that are marked as the accepted solution, and how many?
    gold: {'all': [['usr_009', 'sophie laurent'], ['5']]}
- [C. relational] Which KB articles are referenced in the comment threads of tickets that are not yet closed?
    gold: {'all': [['doc_005', 'aura system requirements'], ['doc_007', 'aura api integration guide'], ['doc_008', 'aura training resources']], 'none': [['doc_001', 'getting started with aura'], ['doc_002', 'aura login troubleshooting'], ['doc_003', 'improving aura response quality'], ['doc_004', 'aura feature overview'], ['doc_006', 'reporting a

In [6]:
import re

# ---------- Graph tools for the agent (each returns JSON the model can read) ----------
def node_view(n):
    """A node as one row, already joined across its edges (what a Cypher RETURN would give)."""
    d = G.nodes[n]; lab = d["label"]
    if lab == "Ticket":
        st = ticket_status(n)
        return {"id": n, "label": lab, "title": d["title"], "status": st["name"], "isClosed": st["isClosed"],
                "category": G.nodes[d["categoryId"]]["name"] if d["categoryId"] in G else None,
                "assignee": user_name(d["assignedToUserId"]), "submitter": user_name(d["createdByUserId"])}
    if lab == "Comment":
        a = in_nbrs(n, "POSTS")[0]
        return {"id": n, "label": lab, "ticket": d["ticketId"], "isSolution": d["isSolution"],
                "author": user_name(a), "author_role": G.nodes[a]["role"], "text": d["content"][:160],
                "references_kb": in_nbrs(n, "REFERENCED_BY")}
    if lab == "User":
        return {"id": n, "label": lab, "name": user_name(n), "role": d["role"]}
    return {"id": n, "label": lab, **{k: v for k, v in d.items() if k in ("name", "title", "summary", "isClosed")}}

def semantic_search(query, label=None, k=5):
    return json.dumps([{"id": c["id"], "score": round(s, 3), "text": c["text"][:300]} for c, s in vector_search(query, k, label)])

def find_nodes(label, filters=None):
    rows = [node_view(n) for n, d in G.nodes(data=True) if d["label"] == label]
    for key, val in (filters or {}).items():
        rows = [r for r in rows if str(r.get(key, "")).lower() == str(val).lower()]
    return json.dumps(rows)

def count_nodes(label, group_by, filters=None):
    """Aggregation done by the graph store (like Cypher count(*)), not by the LLM."""
    groups = {}
    for r in json.loads(find_nodes(label, filters)):
        groups.setdefault(str(r.get(group_by)), []).append(r["id"])
    return json.dumps({k: {"count": len(v), "ids": v} for k, v in sorted(groups.items(), key=lambda kv: -len(kv[1]))})

def get_node(node_id):
    if node_id not in G: return json.dumps({"error": f"no node {node_id}"})
    edges = [{"rel": d["type"], "dir": "out", "node": node_view(v)} for _, v, d in G.out_edges(node_id, data=True)]
    edges += [{"rel": d["type"], "dir": "in", "node": node_view(u)} for u, _, d in G.in_edges(node_id, data=True)]
    return json.dumps({"node": node_view(node_id), "edges": edges})

def get_verified_solution(ticket_id):
    return json.dumps(graph_verified_fix(ticket_id) or {"ticket": ticket_id, "verified_solution": None})

TOOL_IMPL = {"semantic_search": semantic_search, "find_nodes": find_nodes, "count_nodes": count_nodes, "get_node": get_node,
             "get_verified_solution": get_verified_solution}
S = lambda **p: {"type": "object", "properties": p, "required": [next(iter(p))]}
TOOLS = [
    {"type": "function", "function": {"name": "semantic_search", "description": "Vector search over tickets, comments "
     "and KB articles. Optional label: Ticket | Comment | Documentation.",
     "parameters": S(query={"type": "string"}, label={"type": "string"}, k={"type": "integer"})}},
    {"type": "function", "function": {"name": "find_nodes", "description": "Return all nodes of a label (Ticket, Comment, "
     "User, Documentation, Status, Category) as joined rows, optionally filtered by exact field values, e.g. "
     "{'isClosed': 'false', 'category': 'Aura Access'} for Tickets or {'isSolution': 'True'} for Comments.",
     "parameters": S(label={"type": "string"}, filters={"type": "object", "additionalProperties": {"type": "string"}})}},
    {"type": "function", "function": {"name": "count_nodes", "description": "Count nodes of a label grouped by one "
     "field (e.g. Comment grouped by 'author' with {'isSolution': 'True'}; Ticket grouped by 'assignee' with "
     "{'isClosed': 'false'}). Returns count and ids per group. Use this for every 'how many / the most' question.",
     "parameters": S(label={"type": "string"}, group_by={"type": "string"},
                     filters={"type": "object", "additionalProperties": {"type": "string"}})}},
    {"type": "function", "function": {"name": "get_node", "description": "A node with all its incoming/outgoing edges.",
     "parameters": S(node_id={"type": "string"})}},
    {"type": "function", "function": {"name": "get_verified_solution", "description": "For a ticket: if it is closed, "
     "the comment marked as the accepted solution, with its author.", "parameters": S(ticket_id={"type": "string"})}},
]

def graph_agent(question, max_steps=10):
    msgs = [{"role": "system", "content": "You are the Aura IT helpdesk assistant with tools over the helpdesk "
             "knowledge graph. For a new problem report, find the most similar historical tickets (semantic_search with "
             "label='Ticket') and check their verified solutions. Let count_nodes do any counting. " + ANSWER_RULES},
            {"role": "user", "content": question}]
    seen = []
    for _ in range(max_steps):
        msg = chat(msgs, tools=TOOLS).choices[0].message
        if not msg.tool_calls:
            return {"answer": msg.content, "context_ids": seen}
        msgs.append(msg.model_dump(exclude_none=True))
        for tc in msg.tool_calls:
            try:
                out = TOOL_IMPL[tc.function.name](**json.loads(tc.function.arguments))
            except Exception as e:                       # a bad call becomes an observation, not a crash
                out = json.dumps({"error": f"{type(e).__name__}: {e}"})
            seen += re.findall(r"(?:tkt|cmt|doc)_\d{3}", out)
            msgs.append({"role": "tool", "tool_call_id": tc.id, "content": out})
    return {"answer": "(no answer within step budget)", "context_ids": seen}

demo = graph_agent(C_[0][0])
print("Agent demo:", C_[0][0], "\n->", demo["answer"])

Agent demo: Which user has authored the most comments that are marked as the accepted solution, and how many? 
-> The user who has authored the most comments marked as the accepted solution is **Sophie Laurent**, with a total of **5** comments. The IDs of her comments are: cmt_030, cmt_040, cmt_046, cmt_049, and cmt_056.


In [7]:
from concurrent.futures import ThreadPoolExecutor

class Verdict(BaseModel):
    same_fix: bool

def judge_fix(answer, gold_text):
    """Strict LLM judge (family B only): does the answer recommend the same resolution as the verified fix?"""
    v = client.chat.completions.parse(model=CHAT_MODEL, temperature=0, seed=SEED, response_format=Verdict, messages=[
        {"role": "system", "content": "You grade helpdesk answers strictly. same_fix=true only if the candidate's main "
         "recommendation is the same resolution as the reference verified fix (not merely related advice)."},
        {"role": "user", "content": f"Reference verified fix:\n{gold_text}\n\nCandidate answer:\n{answer}"}]).choices[0].message.parsed
    return v.same_fix

def has(ans, aliases): return any(re.search(rf"(?<![\w.]){re.escape(a)}(?![\w])", ans.lower()) for a in aliases)

def score(q, res):
    g, ans = q["gold"], res["answer"]
    if "fix_text" in g:
        return judge_fix(ans, g["fix_text"]), g["comment"] in res["context_ids"]
    ok = all(has(ans, al) for al in g["all"]) and not any(has(ans, al) for al in g.get("none", []))
    return ok, None

def run_one(q):
    out = {}
    for name, fn in (("vector", vector_rag), ("graph", graph_agent)):
        res = fn(q["q"])
        correct, ctx_hit = score(q, res)
        out[name] = {"correct": correct, "ctx_hit": ctx_hit, "answer": res["answer"]}
    return out

with ThreadPoolExecutor(max_workers=8) as pool:
    results = list(pool.map(run_one, questions))

for q, r in zip(questions, results):
    mark = lambda x: "✔" if x else "✘"
    print(f"[{q['family']:<16}] vec {mark(r['vector']['correct'])}  graph {mark(r['graph']['correct'])}  {q['q'][:80]}")

[A. lookup       ] vec ✔  graph ✔  Where can I check Aura's technical requirements and browser compatibility?
[A. lookup       ] vec ✔  graph ✔  Is there a developer guide for calling Aura programmatically?
[A. lookup       ] vec ✔  graph ✔  Where can I find video tutorials on using Aura?
[A. lookup       ] vec ✔  graph ✘  How do I learn to phrase questions so Aura gives more accurate answers?
[A. lookup       ] vec ✔  graph ✔  Which guide explains how to set up a new user account in 3DS systems?
[B. verified fix ] vec ✘  graph ✔  New ticket: I'm unable to access Aura as I'm encountering an error when entering
[B. verified fix ] vec ✘  graph ✔  New ticket: When I inquire about CATIA V6 features, Aura provides outdated infor
[B. verified fix ] vec ✘  graph ✔  New ticket: The response time from Aura has significantly increased, taking over
[B. verified fix ] vec ✔  graph ✔  New ticket: I'm receiving a 'Service Unavailable' error when I try to ask Aura f
[B. verified fix ] vec ✔  graph ✔ 

*Note:* calls use `temperature=0, seed=7`, but OpenAI sampling is not bit-for-bit deterministic and the questions run in
parallel — expect individual cells to flip by about ±1 question between runs. The *pattern* across families is stable.

In [8]:
families = ["A. lookup", "B. verified fix", "C. relational"]
acc = {}
print(f"\n{'Question family':<18} {'n':>3} | {'Vector RAG':>10} | {'Graph agent':>11}")
print("-" * 50)
for fam in families:
    rs = [r for q, r in zip(questions, results) if q["family"] == fam]
    v, g = sum(r["vector"]["correct"] for r in rs), sum(r["graph"]["correct"] for r in rs)
    acc[fam] = (v, g, len(rs))
    print(f"{fam:<18} {len(rs):>3} | {v:>4}/{len(rs):<5} | {g:>5}/{len(rs):<5}")
tv, tg, tn = (sum(x[i] for x in acc.values()) for i in range(3))
print(f"{'Total':<18} {tn:>3} | {tv:>4}/{tn:<5} | {tg:>5}/{tn:<5}")

rb = [r for q, r in zip(questions, results) if q["family"] == "B. verified fix"]
vec_ctx = sum(r["vector"]["ctx_hit"] for r in rb); gra_ctx = sum(r["graph"]["ctx_hit"] for r in rb)

print("\n" + "=" * 70 + "\nRESULTS SUMMARY — Notebook 1 (Aura helpdesk, 122 nodes / 246 edges)\n" + "=" * 70)
for fam in families:
    v, g, n = acc[fam]
    print(f"  {fam:<18} vector {v}/{n} ({v/n:.0%})   graph {g}/{n} ({g/n:.0%})")
print(f"  {'Overall':<18} vector {tv}/{tn} ({tv/tn:.0%})   graph {tg}/{tn} ({tg/tn:.0%})")
print(f"  Verified-fix comment present in context: vector {vec_ctx}/{len(rb)}, graph {gra_ctx}/{len(rb)}")
print(f"  Walkthrough tkt_001: vector answer mentions extension={mentions(vec_walk['answer'], 'extension')}, "
      f"graph answer mentions extension={mentions(graph_walk['answer'], 'extension')}")
print("\nLecture message (Slides 32/36/38): on single-hop lookup a strong vector baseline is as good as the graph;")
print("the graph pays off when the answer depends on structure — status, solution flags, counts and joins")
print("spread across many items. Pick the retriever by question type, and measure against the vector baseline.")


Question family      n | Vector RAG | Graph agent
--------------------------------------------------
A. lookup            5 |    5/5     |     4/5    
B. verified fix     11 |    7/11    |    11/11   
C. relational        6 |    1/6     |     5/6    
Total               22 |   13/22    |    20/22   

RESULTS SUMMARY — Notebook 1 (Aura helpdesk, 122 nodes / 246 edges)
  A. lookup          vector 5/5 (100%)   graph 4/5 (80%)
  B. verified fix    vector 7/11 (64%)   graph 11/11 (100%)
  C. relational      vector 1/6 (17%)   graph 5/6 (83%)
  Overall            vector 13/22 (59%)   graph 20/22 (91%)
  Verified-fix comment present in context: vector 5/11, graph 11/11
  Walkthrough tkt_001: vector answer mentions extension=False, graph answer mentions extension=True

Lecture message (Slides 32/36/38): on single-hop lookup a strong vector baseline is as good as the graph;
the graph pays off when the answer depends on structure — status, solution flags, counts and joins
spread across many ite